# تحليل الطلب على الوجبات | Food Demand Analysis

**الهدف:** مساعدة فريق العمليات والتسويق على فهم أين يتركز الطلب، كيف يختلف بين المراكز والوجبات، وما إذا كانت العروض والخصومات مرتبطة بتغير الطلب.

تُنفّذ هذه الدفترية من البداية إلى النهاية، وتُبقي ملفات المصدر كما هي. المخرجات الرسومية تُحفظ في `reports/figures/`، وجدول قرارات جودة البيانات في `docs/cleaning_decisions.csv`.

**المتطلبات:** Python 3.10+، وJupyter، والحزم `pandas`, `numpy`, `matplotlib`, `scipy`, و`ipython` (يمكن تثبيتها عبر `pip install pandas numpy matplotlib scipy ipython`).

> **تنبيه تحليلي:** البيانات أسبوعية ومجهولة الهوية؛ لا توجد تواريخ أو أسماء مدن حقيقية. الارتباطات والفروق المرصودة لا تثبت السببية، كما أن الطلب المسجل لا يكشف مباشرةً عن نفاد المخزون أو الهدر.


## 1. أسئلة العمل وخطة التحليل

1. كيف يتغير إجمالي الطلب أسبوعيًا؟
2. ما فئات الوجبات والوجبات التي تمثل معظم الطلب، وأي المراكز أعلى أداءً؟
3. هل يرتبط الترويج عبر البريد أو الصفحة الرئيسية بارتفاع الطلب؟ وهل يرتبط الخصم بالطلب؟
4. ما الوجبات التي تجمع بين حجم طلب مرتفع وتذبذب أسبوعي ملحوظ؟

**المنهج:** فحص المخططات والجودة أولًا؛ إثراء بيانات التدريب بمعلومات الوجبات والمراكز؛ بناء مؤشرات الطلب والإيراد والخصم؛ ثم إجراء مقارنات إحصائية مع توضيح حدود الاستدلال.


## 2. تحميل المكتبات والملفات

ضع الملفات الأربعة داخل `data/` وشغّل الدفترية من مجلد المشروع أو من `notebooks/`. يُحمّل `test.csv` أيضًا، لكن لا يحتوي على `num_orders`؛ لذلك لا يُستخدم في حساب مؤشرات الطلب التاريخية أو اختبارات الأثر.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr
from IPython.display import display
# اعداد طريقة عرض النتائج
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
plt.rcParams.update({'figure.figsize': (11, 6), 'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 10})
# تحديد مكان ملفات البيانات
DATA_DIR_CANDIDATES = [Path.cwd() / 'data', Path.cwd().parent / 'data']
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if (path / 'train.csv').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('لم أجد data/train.csv. شغّل الدفترية من مجلد المشروع أو من مجلد notebooks.')
# انشاء مجلدات النتائج
PROJECT_DIR = DATA_DIR.parent
FIGURES_DIR = PROJECT_DIR / 'reports' / 'figures'
DOCS_DIR = PROJECT_DIR / 'docs'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
# قراءة ملفات
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
centers = pd.read_csv(DATA_DIR / 'fulfilment_center_info.csv')
meals = pd.read_csv(DATA_DIR / 'meal_info.csv')

datasets = {'train': train, 'test': test, 'centers': centers, 'meals': meals}
display(pd.DataFrame([{'الملف': name, 'عدد الصفوف': len(frame), 'عدد الأعمدة': frame.shape[1], 'الأعمدة': ', '.join(frame.columns)} for name, frame in datasets.items()]))

,الملف,عدد الصفوف,عدد الأعمدة,الأعمدة
0,train,456548,9,"id, week, center_id, meal_id, checkout_price, ..."
1,test,32573,8,"id, week, center_id, meal_id, checkout_price, ..."
2,centers,77,5,"center_id, city_code, region_code, center_type..."
3,meals,51,3,"meal_id, category, cuisine"


## 3. فهم بنية البيانات والتحقق من المفاتيح

المستوى المتوقع للتدريب هو سجل لكل **مركز × وجبة × أسبوع**. نتحقق من الأعمدة والمفاتيح قبل الدمج؛ ونستخدم `validate='many_to_one'` حتى يفشل الدمج إذا كانت جداول الأبعاد غير فريدة بدلًا من مضاعفة الطلب بصمت.


In [7]:
expected_train = {'id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders'}
expected_test = expected_train - {'num_orders'}
expected_centers = {'center_id', 'city_code', 'region_code', 'center_type', 'op_area'}
expected_meals = {'meal_id', 'category', 'cuisine'}
for label, frame, required in [('train', train, expected_train), ('test', test, expected_test), ('centers', centers, expected_centers), ('meals', meals, expected_meals)]:
    missing_columns = required - set(frame.columns)
    if missing_columns:
        raise ValueError(f'{label}: أعمدة مطلوبة غير موجودة: {sorted(missing_columns)}')

print('مفاتيح التكرار في جداول الأبعاد:')
key_checks = pd.DataFrame([
    {'الجدول': 'centers', 'مفتاح التحقق': 'center_id', 'تكرارات المفتاح': int(centers['center_id'].duplicated().sum())},
    {'الجدول': 'meals', 'مفتاح التحقق': 'meal_id', 'تكرارات المفتاح': int(meals['meal_id'].duplicated().sum())},
    {'الجدول': 'train', 'مفتاح التحقق': '(week, center_id, meal_id)', 'تكرارات المفتاح': int(train.duplicated(['week', 'center_id', 'meal_id']).sum())},
    {'الجدول': 'test', 'مفتاح التحقق': '(week, center_id, meal_id)', 'تكرارات المفتاح': int(test.duplicated(['week', 'center_id', 'meal_id']).sum())},
])
display(key_checks)
if centers['center_id'].duplicated().any() or meals['meal_id'].duplicated().any():
    raise ValueError('مفاتيح جداول الأبعاد ليست فريدة؛ أوقفنا التحليل منعًا لمضاعفة الصفوف عند الدمج.')

print(f"نطاق الأسابيع في التدريب: {train['week'].min()}–{train['week'].max()} | الاختبار: {test['week'].min()}–{test['week'].max()}")
print(f"هل يحتوي test.csv على num_orders؟ {'نعم' if 'num_orders' in test.columns else 'No'}")

مفاتيح التكرار في جداول الأبعاد:


,الجدول,مفتاح التحقق,تكرارات المفتاح
0,centers,center_id,0
1,meals,meal_id,0
2,train,"(week, center_id, meal_id)",0
3,test,"(week, center_id, meal_id)",0


نطاق الأسابيع في التدريب: 1–145 | الاختبار: 146–155
هل يحتوي test.csv على num_orders؟ No


## 4. جودة البيانات وقرارات التنظيف

لا نحذف القيم الشاذة لمجرد بُعدها عن المتوسط: الطلب المرتفع قد يمثل ذروة تشغيلية حقيقية. كذلك قد يكون سعر البيع أعلى من السعر الأساسي حالة بيانات أو تسعير؛ نُبلغ عنها ولا نصححها دون دليل. نفحص القيم المفقودة والمكررة والأسعار والطلبات والأسابيع والتغطية الزمنية.


In [ ]:
key_cols = ['week', 'center_id', 'meal_id']
missing_summary = pd.DataFrame([
    {'الجدول': name, 'العمود': column, 'المفقود': int(frame[column].isna().sum()), 'النسبة %': frame[column].isna().mean() * 100}
    for name, frame in datasets.items() for column in frame.columns if frame[column].isna().any()
])
if missing_summary.empty:
    print('لا توجد قيم مفقودة في الملفات الأربعة.')
else:
    display(missing_summary.sort_values('المفقود', ascending=False))

duplicate_key_count = int(train.duplicated(key_cols).sum())
duplicate_test_keys = int(test.duplicated(key_cols).sum())
above_base = int((train['checkout_price'] > train['base_price']).sum())
nonpositive_prices = int(((train['checkout_price'] <= 0) | (train['base_price'] <= 0)).sum())
negative_orders = int((train['num_orders'] < 0).sum())
unknown_center = int((~train['center_id'].isin(centers['center_id'])).sum())
unknown_meal = int((~train['meal_id'].isin(meals['meal_id'])).sum())
week_values = set(train['week'].dropna().astype(int).unique())
expected_weeks = set(range(int(train['week'].min()), int(train['week'].max()) + 1))
missing_weeks = sorted(expected_weeks - week_values)
series_weeks = train.groupby(['center_id', 'meal_id'])['week'].nunique()
incomplete_series = int((series_weeks < train['week'].nunique()).sum())
q1, q3 = train['num_orders'].quantile([0.25, 0.75])
iqr = q3 - q1
high_order_iqr = int((train['num_orders'] > q3 + 1.5 * iqr).sum())

quality_checks = pd.DataFrame([
    {'الفحص': 'صفوف تدريب مكررة بحسب الأسبوع/المركز/الوجبة', 'العدد': duplicate_key_count},
    {'الفحص': 'صفوف اختبار مكررة بحسب الأسبوع/المركز/الوجبة', 'العدد': duplicate_test_keys},
    {'الفحص': 'سعر بيع أعلى من السعر الأساسي', 'العدد': above_base},
    {'الفحص': 'سعر بيع أو أساسي غير موجب', 'العدد': nonpositive_prices},
    {'الفحص': 'طلبات سالبة', 'العدد': negative_orders},
    {'الفحص': 'صفوف تدريب بمركز غير معروف', 'العدد': unknown_center},
    {'الفحص': 'صفوف تدريب بوجبة غير معروفة', 'العدد': unknown_meal},
    {'الفحص': 'أسابيع مفقودة من النطاق العام', 'العدد': len(missing_weeks)},
    {'الفحص': 'سلاسل مركز-وجبة لا تغطي كل أسابيع التدريب', 'العدد': incomplete_series},
    {'الفحص': 'طلبات فوق حد IQR الأعلى (للمراجعة فقط)', 'العدد': high_order_iqr},
])
display(quality_checks)
print('الأسابيع غير الموجودة ضمن نطاق التدريب:', missing_weeks if missing_weeks else 'لا يوجد')

In [ ]:
cleaning_decisions = pd.DataFrame([
    {'المشكلة': 'القيم المفقودة', 'العدد': int(sum(frame.isna().sum().sum() for frame in datasets.values())), 'القرار': 'إظهار الأعداد؛ عدم التعويض تلقائيًا. تُراجع الأعمدة المتأثرة قبل أي نموذج.'},
    {'المشكلة': 'مفاتيح تدريب مكررة (أسبوع/مركز/وجبة)', 'العدد': duplicate_key_count, 'القرار': 'الإبلاغ والاحتفاظ بالسجلات الأصلية؛ لا حذف قبل تحديد سبب التكرار.'},
    {'المشكلة': 'سعر البيع أعلى من السعر الأساسي', 'العدد': above_base, 'القرار': 'الاحتفاظ به؛ لا يُفترض أن كل فرق سعر خصم.'},
    {'المشكلة': 'أسعار غير موجبة أو طلبات سالبة', 'العدد': nonpositive_prices + negative_orders, 'القرار': 'الإبلاغ وعدم تعديل المصدر؛ استبعادها فقط من المؤشر الذي يتطلب قيمة موجبة مع توثيق ذلك.'},
    {'المشكلة': 'أسابيع مفقودة على مستوى البيانات', 'العدد': len(missing_weeks), 'القرار': 'عدم إنشاء أسابيع صناعية؛ تمييز غياب السجل عن طلب صفري.'},
    {'المشكلة': 'سلاسل مركز-وجبة غير مكتملة', 'العدد': incomplete_series, 'القرار': 'عدم ملء الأسابيع المفقودة بصفر؛ قد يعني غياب العرض أو السجل لا انعدام الطلب.'},
    {'المشكلة': 'قيم طلب مرتفعة وفق حد IQR', 'العدد': high_order_iqr, 'القرار': 'اعتبارها إشارات للمراجعة لا أخطاء مؤكدة؛ الإبقاء عليها في التحليل.'},
])
cleaning_path = DOCS_DIR / 'cleaning_decisions.csv'
cleaning_decisions.to_csv(cleaning_path, index=False, encoding='utf-8-sig')
display(cleaning_decisions)
print(f'حُفظ سجل قرارات التنظيف في: {cleaning_path.relative_to(PROJECT_DIR)}')

## 5. إثراء البيانات وربط الجداول

نضيف خصائص المركز والوجبة إلى التدريب والاختبار بمفتاحيهما. نتحقق بعد الدمج من ثبات عدد الصفوف وإجمالي الطلب؛ أي تغير يعني أن عملية الربط غير آمنة. ملفات `data/` للقراءة فقط.


In [8]:
train_rows_before = len(train)
orders_before = train['num_orders'].sum()
train_enriched = train.merge(centers, on='center_id', how='left', validate='many_to_one').merge(meals, on='meal_id', how='left', validate='many_to_one')
test_enriched = test.merge(centers, on='center_id', how='left', validate='many_to_one').merge(meals, on='meal_id', how='left', validate='many_to_one')
assert len(train_enriched) == train_rows_before, 'تغير عدد صفوف التدريب بعد الربط.'
assert train_enriched['num_orders'].sum() == orders_before, 'تغير إجمالي الطلب بعد الربط.'
print(f'التدريب: {train_rows_before:,} صف قبل الربط وبعده؛ الطلب قبل/بعد = {orders_before:,.0f} / {train_enriched["num_orders"].sum():,.0f}.')
print(f'الاختبار: {len(test):,} صف قبل الربط و{len(test_enriched):,} بعده؛ عمود num_orders غير موجود ولا يُنشأ.')
print('قيم أبعاد مفقودة بعد الربط:', int(train_enriched[['center_type', 'category', 'cuisine']].isna().any(axis=1).sum()))

التدريب: 456,548 صف قبل الربط وبعده؛ الطلب قبل/بعد = 119,557,485 / 119,557,485.
الاختبار: 32,573 صف قبل الربط و32,573 بعده؛ عمود num_orders غير موجود ولا يُنشأ.
قيم أبعاد مفقودة بعد الربط: 0


## 6. بناء المؤشرات

- **الإيراد التقريبي:** سعر البيع × عدد الطلبات؛ لا يساوي الربح ولا يأخذ التكاليف في الحسبان.
- **نسبة الخصم:** (السعر الأساسي − سعر البيع) ÷ السعر الأساسي؛ قد تكون سالبة إذا تجاوز سعر البيع الأساسي.
- **الترويج:** البريد والظهور في الصفحة الرئيسية قناتان منفصلتان؛ و`any_promo` تعني وجود واحدة على الأقل.
- تُحسب المؤشرات من سجلات التدريب التي يتوفر فيها الهدف فقط.


In [ ]:
df = train_enriched.copy()
df['revenue_estimate'] = df['checkout_price'] * df['num_orders']
df['discount_pct'] = (df['base_price'] - df['checkout_price']) / df['base_price'].where(df['base_price'] > 0)
df['email_promo'] = df['emailer_for_promotion'].eq(1)
df['homepage_promo'] = df['homepage_featured'].eq(1)
df['any_promo'] = df['email_promo'] | df['homepage_promo']

weekly = df.groupby('week', as_index=False).agg(orders=('num_orders', 'sum'), revenue_estimate=('revenue_estimate', 'sum'), active_center_meal_pairs=('meal_id', 'size'))
total_orders = int(df['num_orders'].sum())
weekly_mean = weekly['orders'].mean()
kpi_table = pd.DataFrame([
    {'المؤشر': 'إجمالي الطلبات المسجلة', 'القيمة': f'{total_orders:,.0f}'},
    {'المؤشر': 'متوسط الطلب الأسبوعي', 'القيمة': f'{weekly_mean:,.0f}'},
    {'المؤشر': 'الأسابيع المغطاة', 'القيمة': f'{df["week"].nunique():,}'},
    {'المؤشر': 'المراكز الفريدة', 'القيمة': f'{df["center_id"].nunique():,}'},
    {'المؤشر': 'الوجبات الفريدة', 'القيمة': f'{df["meal_id"].nunique():,}'},
    {'المؤشر': 'الإيراد التقديري (وحدة سعر البيانات)', 'القيمة': f'{df["revenue_estimate"].sum():,.0f}'},
    {'المؤشر': 'متوسط نسبة الخصم', 'القيمة': f'{df["discount_pct"].mean():.1%}'},
    {'المؤشر': 'نسبة الصفوف التي عليها ترويج', 'القيمة': f'{df["any_promo"].mean():.1%}'},
])
display(kpi_table)

## 7. س1 — اتجاه الطلب الأسبوعي

كل نقطة هي مجموع الطلبات المسجلة في أسبوع واحد. العنوان يبرز الفرق بين أعلى وأدنى أسبوع في العينة؛ لا ينبغي تفسيره كاتجاه مستقبلي دون نموذج وتحقق زمني.


In [ ]:
peak_week = weekly.loc[weekly['orders'].idxmax()]
low_week = weekly.loc[weekly['orders'].idxmin()]
spread_pct = (peak_week['orders'] / low_week['orders'] - 1) * 100 if low_week['orders'] else np.nan
fig, ax = plt.subplots()
ax.plot(weekly['week'], weekly['orders'], color='#176B87', linewidth=2)
ax.scatter([peak_week['week'], low_week['week']], [peak_week['orders'], low_week['orders']], color=['#2E8B57', '#D97706'], zorder=3)
ax.set(title=f"Peak week {int(peak_week['week'])} is {spread_pct:.0f}% above the lowest week", xlabel='Week number', ylabel='Total recorded orders')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '01_weekly_demand.png', bbox_inches='tight')
plt.show()
print(f"الأعلى: الأسبوع {int(peak_week['week'])} ({peak_week['orders']:,.0f}) | الأدنى: الأسبوع {int(low_week['week'])} ({low_week['orders']:,.0f}).")

## 8. س2 — الفئات والوجبات التي تمثل معظم الطلب

يُرتب تحليل باريتو الوجبات حسب مجموع الطلب، ثم يحدد عدد الوجبات اللازمة لتجاوز 80% تراكميًا. هذه حصة من **الطلبات المسجلة** وليست حصة الأرباح أو الهدر.


In [ ]:
category_orders = df.groupby('category', as_index=False).agg(orders=('num_orders', 'sum')).sort_values('orders', ascending=False)
category_orders['share_pct'] = category_orders['orders'] / total_orders * 100
meal_orders = df.groupby(['meal_id', 'category', 'cuisine'], as_index=False).agg(orders=('num_orders', 'sum')).sort_values('orders', ascending=False)
meal_orders['cumulative_share'] = meal_orders['orders'].cumsum() / total_orders
meals_to_80 = int((meal_orders['cumulative_share'] < 0.80).sum() + 1)
top_category = category_orders.iloc[0]
print(f"الفئة الأعلى: {top_category['category']} — {top_category['share_pct']:.1f}% من الطلبات.")
print(f'عدد الوجبات اللازمة لتجاوز 80% من الطلبات: {meals_to_80} من أصل {len(meal_orders)} وجبة.')
display(category_orders)
display(meal_orders.head(10))

fig, ax = plt.subplots()
ax.bar(category_orders['category'], category_orders['orders'], color='#3399A8')
ax.set(title=f"{top_category['category']} leads category demand ({top_category['share_pct']:.1f}% of orders)", xlabel='Meal category', ylabel='Total recorded orders')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '02_demand_by_category.png', bbox_inches='tight')
plt.show()

## 9. س3 — أداء مراكز التجهيز

نقارن مجموع الطلبات ومتوسط الطلب لكل سجل حسب نوع المركز. المجموع يتأثر بحجم المركز وعدد الوجبات/الأسابيع المتاحة؛ لذلك لا يعني وحده أن نوعًا ما أكثر كفاءة. يوضح جدول المراكز صاحبة أعلى حجم طلب أيضًا.


In [ ]:
center_orders = df.groupby(['center_id', 'center_type'], as_index=False).agg(orders=('num_orders', 'sum'), weekly_mean=('num_orders', 'mean'), observations=('num_orders', 'size'), op_area=('op_area', 'first'))
center_type_summary = df.groupby('center_type', as_index=False).agg(total_orders=('num_orders', 'sum'), mean_orders_per_record=('num_orders', 'mean'), centers=('center_id', 'nunique'))
top_center = center_orders.sort_values('orders', ascending=False).iloc[0]
display(center_type_summary.sort_values('total_orders', ascending=False))
display(center_orders.sort_values('orders', ascending=False).head(10))
print(f"أعلى مركز في مجموع الطلب المسجل: {int(top_center['center_id'])} ({top_center['center_type']}) بعدد {top_center['orders']:,.0f} طلب.")

plot_centers = center_type_summary.sort_values('total_orders', ascending=False)
fig, ax = plt.subplots()
ax.bar(plot_centers['center_type'], plot_centers['total_orders'], color='#6A8EAE')
ax.set(title='Center types differ in total recorded demand; size and coverage also matter', xlabel='Fulfilment center type', ylabel='Total recorded orders')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '03_center_type.png', bbox_inches='tight')
plt.show()

## 10. س4 — مقارنة الطلب مع الترويج

لتقليل اختلاف مزيج المنتجات، نحتفظ فقط بسلاسل **المركز × الوجبة** التي ظهرت فيها مشاهدات مع الترويج وبدونه. نقارن توزيعات الطلب باختبار Mann–Whitney ثنائي الطرف، ونعرض الوسيطين والفرق النسبي والقيم الاحتمالية قبل وبعد تصحيح Holm للمقارنات الثلاث. هذا اختبار ارتباط على مشاهدات غير عشوائية، لا تجربة سببية؛ كما أن تداخل الحملات وارتباط السجلات عبر الزمن يحدّان من تفسير قيمة p.


In [ ]:
def matched_promotion_comparison(data: pd.DataFrame, flag: str, label: str) -> dict:
    pair_counts = data.groupby(['center_id', 'meal_id'])[flag].nunique()
    matched_pairs = pair_counts[pair_counts == 2].index
    matched_index = pd.MultiIndex.from_frame(data[['center_id', 'meal_id']])
    eligible = data.loc[matched_index.isin(matched_pairs), [flag, 'num_orders']]
    promoted = eligible.loc[eligible[flag], 'num_orders']
    unpromoted = eligible.loc[~eligible[flag], 'num_orders']
    if promoted.empty or unpromoted.empty:
        return {'القناة': label, 'سلاسل مركز-وجبة المتطابقة': len(matched_pairs), 'عدد صفوف الترويج': len(promoted), 'عدد الصفوف دون ترويج': len(unpromoted), 'وسيط الطلب مع الترويج': np.nan, 'وسيط الطلب دون ترويج': np.nan, 'الفرق النسبي %': np.nan, 'p-value': np.nan}
    p_value = mannwhitneyu(promoted, unpromoted, alternative='two-sided', method='auto').pvalue
    median_promoted, median_unpromoted = promoted.median(), unpromoted.median()
    relative_change = (median_promoted / median_unpromoted - 1) * 100 if median_unpromoted else np.nan
    return {'القناة': label, 'سلاسل مركز-وجبة المتطابقة': len(matched_pairs), 'عدد صفوف الترويج': len(promoted), 'عدد الصفوف دون ترويج': len(unpromoted), 'وسيط الطلب مع الترويج': median_promoted, 'وسيط الطلب دون ترويج': median_unpromoted, 'الفرق النسبي %': relative_change, 'p-value': p_value}

promotion_results = pd.DataFrame([
    matched_promotion_comparison(df, 'email_promo', 'البريد الترويجي'),
    matched_promotion_comparison(df, 'homepage_promo', 'الظهور في الصفحة الرئيسية'),
    matched_promotion_comparison(df, 'any_promo', 'أي ترويج'),
])
def holm_adjust(p_values: pd.Series) -> pd.Series:
    values = p_values.to_numpy(dtype=float)
    adjusted = np.full(values.shape, np.nan)
    valid_positions = np.flatnonzero(np.isfinite(values))
    if len(valid_positions):
        order = np.argsort(values[valid_positions])
        sorted_values = values[valid_positions][order]
        scaled = (len(sorted_values) - np.arange(len(sorted_values))) * sorted_values
        adjusted_sorted = np.minimum(1, np.maximum.accumulate(scaled))
        adjusted[valid_positions[order]] = adjusted_sorted
    return pd.Series(adjusted, index=p_values.index)

promotion_results['p-value (Holm)'] = holm_adjust(promotion_results['p-value'])
def format_p_value(value: float) -> str:
    if pd.isna(value):
        return 'غير متاح'
    return '< 1e-300 (تقريب عددي)' if value == 0 else f'{value:.3g}'

promotion_display = promotion_results.copy()
promotion_display['p-value'] = promotion_display['p-value'].map(format_p_value)
promotion_display['p-value (Holm)'] = promotion_display['p-value (Holm)'].map(format_p_value)
display(promotion_display)

pair_counts = df.groupby(['center_id', 'meal_id'])['any_promo'].nunique()
matched_pairs = pair_counts[pair_counts == 2].index
row_pairs = pd.MultiIndex.from_frame(df[['center_id', 'meal_id']])
plot_data = df.loc[row_pairs.isin(matched_pairs)].copy()
plot_data['Promotion status'] = np.where(plot_data['any_promo'], 'Any promotion', 'No promotion')
fig, ax = plt.subplots()
ax.boxplot([plot_data.loc[plot_data['Promotion status'] == group, 'num_orders'].sample(min(12000, (plot_data['Promotion status'] == group).sum()), random_state=42) for group in ['No promotion', 'Any promotion']], tick_labels=['No promotion', 'Any promotion'], showfliers=False)
ax.set(title='Order distributions by promotion status within matched center-meal pairs', xlabel='Promotion status', ylabel='Orders per record')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '04_promotion_comparison.png', bbox_inches='tight')
plt.show()

## 11. س5 — الخصم وعلاقته بالطلب

نستخدم ارتباط Spearman لأنه يقيس العلاقة الرتيبة ولا يشترط علاقة خطية. الاستدلال يظل وصفيًا: قد تُمنح الخصومات عمدًا للوجبات أو الأسابيع ذات خصائص مختلفة، وقد تتزامن مع البريد والظهور في الصفحة.


In [ ]:
discount_data = df[['discount_pct', 'num_orders']].replace([np.inf, -np.inf], np.nan).dropna()
spearman_result = spearmanr(discount_data['discount_pct'], discount_data['num_orders'])
rho, spearman_p = float(spearman_result.statistic), float(spearman_result.pvalue)
print(f'ارتباط Spearman = {rho:.3f} | p-value = {format_p_value(spearman_p)} | n = {len(discount_data):,}.')
sample = df.loc[df['discount_pct'].notna()].sample(min(12000, df['discount_pct'].notna().sum()), random_state=42)
fig, ax = plt.subplots()
ax.scatter(sample['discount_pct'] * 100, sample['num_orders'], alpha=0.22, s=12, color='#7B4F9D', edgecolors='none')
ax.set(title=f'Discount and order volume have a Spearman association of {rho:.2f} (not causal)', xlabel='Discount from base price (%)', ylabel='Orders per record')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '05_discount_relationship.png', bbox_inches='tight')
plt.show()

## 12. س6 — التذبذب وحجم الطلب حسب الوجبة

نجمّع الطلب الأسبوعي لكل وجبة عبر المراكز، ثم نحسب معامل الاختلاف (الانحراف المعياري ÷ المتوسط). نرتب مزيج حجم الطلب ومعامل الاختلاف باستخدام متوسط الرتبتين المئويتين؛ المؤشر أداة **فرز للمراجعة** وليس كمية شراء مثلى. غياب سجل أسبوعي لا يُعامل كصفر.


In [ ]:
meal_weekly = df.groupby(['meal_id', 'week'], as_index=False).agg(weekly_orders=('num_orders', 'sum'))
meal_volatility = meal_weekly.groupby('meal_id', as_index=False).agg(total_orders=('weekly_orders', 'sum'), mean_weekly_orders=('weekly_orders', 'mean'), std_weekly_orders=('weekly_orders', 'std'), weeks_observed=('week', 'nunique'))
meal_volatility['cv'] = meal_volatility['std_weekly_orders'] / meal_volatility['mean_weekly_orders'].replace(0, np.nan)
meal_volatility = meal_volatility.merge(meals, on='meal_id', how='left', validate='one_to_one')
meal_volatility['volume_percentile'] = meal_volatility['total_orders'].rank(pct=True)
meal_volatility['volatility_percentile'] = meal_volatility['cv'].rank(pct=True)
meal_volatility['review_priority_score'] = meal_volatility[['volume_percentile', 'volatility_percentile']].mean(axis=1)
meal_volatility = meal_volatility.sort_values('review_priority_score', ascending=False)
display(meal_volatility.head(15)[['meal_id', 'category', 'cuisine', 'total_orders', 'mean_weekly_orders', 'cv', 'weeks_observed', 'review_priority_score']])

top_volatile = meal_volatility.head(10).sort_values('review_priority_score')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_volatile['meal_id'].astype(str), top_volatile['review_priority_score'], color='#D97706')
ax.set(title='Top meals combine high total demand with high weekly variability', xlabel='Mean percentile rank: demand volume and variability', ylabel='Meal ID')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '06_meal_volatility.png', bbox_inches='tight')
plt.show()

series_demand = df.groupby(['center_id', 'meal_id'], as_index=False).agg(mean_weekly_orders=('num_orders', 'mean'), p90_weekly_orders=('num_orders', lambda values: values.quantile(0.90)), std_weekly_orders=('num_orders', 'std'), observed_weeks=('week', 'nunique'))
series_demand['cv'] = series_demand['std_weekly_orders'] / series_demand['mean_weekly_orders'].replace(0, np.nan)
print('أعلى سلاسل مركز-وجبة حسب متوسط الطلب ومعيار 90% التخطيطي:')
display(series_demand.sort_values('mean_weekly_orders', ascending=False).head(10).merge(meals, on='meal_id', validate='many_to_one'))

## 13. إضافة اختيارية — تنبؤ الطلب ومقارنة MAE

نستخدم انحدارًا خطيًا بسيطًا بـ NumPy (دون حزمة إضافية) مع الطلب من الأسبوع السابق، والأسبوع، والخصم، ومؤشري الترويج. الاختبار الزمني محجوز للأسابيع 136–145؛ يُدرّب النموذج على الأسابيع السابقة، ويُقارن بخط أساس «طلب الأسبوع السابق» على الصفوف التي لها أسبوع سابق متصل. هذا تقييم خطوة واحدة إلى الأمام، ولا يضمن جودة تنبؤ عشرة أسابيع متتابعة. بعد ذلك يُعاد التدريب على كامل التدريب لإنتاج تنبؤات `test.csv`، مع استخدام آخر طلب معروف عند فجوة زمنية وحفظ التنبؤات دون كتابة `num_orders` إلى بيانات الاختبار.


In [ ]:
model_features = ['lag1_orders', 'week', 'discount_pct', 'email_promo', 'homepage_promo']
ordered = df.sort_values(['center_id', 'meal_id', 'week']).copy()
series_keys = ['center_id', 'meal_id']
grouped = ordered.groupby(series_keys, sort=False)
ordered['lag1_orders'] = grouped['num_orders'].shift(1)
ordered['previous_week'] = grouped['week'].shift(1)
one_week_lag = ordered.loc[ordered['previous_week'].eq(ordered['week'] - 1)].copy()
one_week_lag = one_week_lag.dropna(subset=model_features + ['num_orders'])
validation_start = int(df['week'].max()) - 9
fit_rows = one_week_lag.loc[one_week_lag['week'] < validation_start]
validation_rows = one_week_lag.loc[one_week_lag['week'] >= validation_start]
if fit_rows.empty or validation_rows.empty:
    raise ValueError('لا توجد سجلات كافية لتقسيم التدريب الزمني والتحقق من الأسابيع العشرة الأخيرة.')

def fit_linear_regression(training: pd.DataFrame):
    means = training[model_features].mean().to_numpy(dtype=float)
    scales = training[model_features].std(ddof=0).replace(0, 1).to_numpy(dtype=float)
    x = (training[model_features].to_numpy(dtype=float) - means) / scales
    design = np.column_stack([np.ones(len(x)), x])
    coefficients = np.linalg.lstsq(design, training['num_orders'].to_numpy(dtype=float), rcond=None)[0]
    return means, scales, coefficients

def predict_linear_regression(frame: pd.DataFrame, parameters) -> np.ndarray:
    means, scales, coefficients = parameters
    x = (frame[model_features].to_numpy(dtype=float) - means) / scales
    return np.maximum(0, np.column_stack([np.ones(len(x)), x]) @ coefficients)

validation_model = fit_linear_regression(fit_rows)
validation_prediction = predict_linear_regression(validation_rows, validation_model)
model_mae = float(np.mean(np.abs(validation_rows['num_orders'].to_numpy() - validation_prediction)))
naive_mae = float(np.mean(np.abs(validation_rows['num_orders'].to_numpy() - validation_rows['lag1_orders'].to_numpy())))
mae_comparison = pd.DataFrame([
    {'النموذج': 'انحدار خطي (خطوة واحدة)', 'MAE': model_mae, 'صفوف التحقق': len(validation_rows)},
    {'النموذج': 'خط أساس: طلب الأسبوع السابق', 'MAE': naive_mae, 'صفوف التحقق': len(validation_rows)},
])
display(mae_comparison)
print(f'نطاق التحقق: الأسابيع {validation_rows["week"].min()}–{validation_rows["week"].max()}؛ قيّمنا فقط أزواج المركز-الوجبة ذات أسبوع سابق متصل.')

final_model = fit_linear_regression(one_week_lag)
last_actual = ordered.groupby(series_keys, sort=False).tail(1).set_index(series_keys)['num_orders'].to_dict()
meal_fallback = df.groupby('meal_id')['num_orders'].mean().to_dict()
prediction_records = []
future_rows = test_enriched.sort_values(['week', 'center_id', 'meal_id']).copy()
for future_week, week_rows in future_rows.groupby('week', sort=True):
    week_features = []
    week_keys = []
    for row in week_rows.itertuples(index=False):
        key = (row.center_id, row.meal_id)
        lag_value = last_actual.get(key, meal_fallback.get(row.meal_id, 0.0))
        discount_value = (row.base_price - row.checkout_price) / row.base_price if row.base_price > 0 else 0.0
        week_features.append({'lag1_orders': lag_value, 'week': row.week, 'discount_pct': discount_value, 'email_promo': bool(row.emailer_for_promotion), 'homepage_promo': bool(row.homepage_featured)})
        week_keys.append((row.id, key))
    feature_frame = pd.DataFrame(week_features, columns=model_features)
    linear_orders = predict_linear_regression(feature_frame, final_model)
    for (row_id, key), row_features, linear_prediction in zip(week_keys, week_features, linear_orders):
        previous_week_prediction = row_features['lag1_orders']
        selected_model = 'linear_regression' if model_mae < naive_mae else 'previous_week_baseline'
        selected_prediction = linear_prediction if model_mae < naive_mae else previous_week_prediction
        prediction_records.append({'id': row_id, 'week': future_week, 'center_id': key[0], 'meal_id': key[1], 'linear_regression_orders': linear_prediction, 'previous_week_baseline_orders': previous_week_prediction, 'selected_forecast_model': selected_model, 'predicted_num_orders': selected_prediction})
        last_actual[key] = selected_prediction

test_predictions = pd.DataFrame(prediction_records)
assert len(test_predictions) == len(test), 'تنبؤ واحد لكل صف من test.csv متوقعًا.'
assert test_predictions['id'].is_unique, 'معرفات التنبؤ يجب أن تكون فريدة.'
predictions_path = PROJECT_DIR / 'reports' / 'test_predictions.csv'
test_predictions.to_csv(predictions_path, index=False)
print(f'حُفظت {len(test_predictions):,} تنبؤات اختبار في {predictions_path.relative_to(PROJECT_DIR)}.')
print(f'النموذج المختار للتنبؤات هو {test_predictions["selected_forecast_model"].iloc[0]} بناءً على MAE الأسابيع العشرة المحجوزة.')
print('التنبؤات خطوة للأمام وبترحيل توقعات الأسابيع السابقة؛ تحقّق من MAE زمني أحدث قبل استخدامها في قرار شراء.')
display(test_predictions.head())

## 14. ملخص قابل للتواصل مع صاحب العمل

تُنشأ النقاط التالية من البيانات بعد إعادة التشغيل، كي لا تُكتب أرقام ثابتة قد تصبح قديمة إذا تغيرت الملفات. افصل بين النتائج التي تقيسها البيانات والتفسير أو الإجراء المقترح.


In [ ]:
top_meal = meal_orders.iloc[0]
email_result = promotion_results.loc[promotion_results['القناة'] == 'البريد الترويجي'].iloc[0]
homepage_result = promotion_results.loc[promotion_results['القناة'] == 'الظهور في الصفحة الرئيسية'].iloc[0]
highest_priority_meal = meal_volatility.iloc[0]
print('أبرز النتائج المرصودة:')
print(f"1) سُجل {total_orders:,.0f} طلبًا خلال {df['week'].nunique()} أسبوعًا؛ متوسط الأسبوع {weekly_mean:,.0f} طلب.")
print(f"2) الفئة الأولى هي {top_category['category']} ({top_category['share_pct']:.1f}% من الطلبات)، والوجبة الأعلى {int(top_meal['meal_id'])} ({top_meal['orders']:,.0f} طلب).")
print(f"3) ارتبطت الخصومات بالطلب بمعامل Spearman قدره {rho:.3f} (p {format_p_value(spearman_p)})؛ هذه علاقة لا تثبت أثرًا سببيًا.")
for result in [email_result, homepage_result]:
    print(f"4) {result['القناة']}: فرق الوسيط {result['الفرق النسبي %']:.1f}%، p-Holm {format_p_value(result['p-value (Holm)'])}، ضمن سلاسل مركز-وجبة متطابقة.")
print(f"5) أعلى وجبة في أولوية مراجعة الحجم والتذبذب: {int(highest_priority_meal['meal_id'])} ({highest_priority_meal['category']}); راجع نتائجها محليًا حسب المركز قبل قرار المخزون.")

print('\nتوصيات تشغيلية قابلة للقياس:')
print('• ابدأ بتجربة مخزون تجريبية للوجبات ذات حجم الطلب والتذبذب المرتفعين، مع استخدام المئين 90 لكل مركز-وجبة كخط أساس مبدئي؛ قِس نسبة نفاد المخزون والهدر أسبوعيًا مقارنة بخط الأساس.')
print(f"• اختبر حملات البريد والصفحة الرئيسية بتجربة عشوائية/متدرجة؛ فرق الوسيط المرصود للبريد {email_result['الفرق النسبي %']:.1f}% وللصفحة {homepage_result['الفرق النسبي %']:.1f}%، وليس تقديرًا سببيًا؛ قِس الزيادة في الطلب والهامش بعد تكلفة الحملة.")
print(f'• نفّذ اختبار تسعير مضبوطًا بدل تعميم علاقة الخصم (Spearman={rho:.3f})؛ قِس الطلب والهامش لكل وجبة-مركز مع تسجيل الترويج المصاحب.')

print('\nحدود يجب ذكرها في التقرير: لا تواريخ فعلية أو مدن مكشوفة؛ الطلب المرصود قد يكون مقيدًا بالمخزون؛ لا بيانات عن التلف أو تكلفة المواد أو الهامش؛ الترويج والأسعار غير عشوائيين؛ الارتباط واختبار Mann–Whitney لا يثبتان السببية.')

## 15. التحقق من المخرجات

تُحفظ الرسوم الستة بصيغة PNG وجدول قرارات التنظيف CSV. يعرض الفحص أسماء الملفات وأحجامها؛ يمكن إدراج الصور مباشرةً في تقرير العميل أو العرض.


In [ ]:
figure_files = sorted(FIGURES_DIR.glob('*.png'))
output_check = pd.DataFrame([{'الملف': str(path.relative_to(PROJECT_DIR)), 'الحجم KB': round(path.stat().st_size / 1024, 1)} for path in figure_files])
display(output_check)
assert len(figure_files) >= 4, 'المطلوب حفظ أربعة رسوم PNG على الأقل.'
assert cleaning_path.is_file(), 'لم يُحفظ جدول قرارات التنظيف.'
assert predictions_path.is_file() and len(test_predictions) == len(test), 'يجب حفظ تنبؤ لكل صف اختبار.'
assert 'num_orders' not in test_enriched.columns, 'يجب عدم اختلاق هدف لبيانات الاختبار.'
print('اكتمل التحليل والتحقق من المخرجات بنجاح.')